# Machine Learning Foundation: MLOps, Training Pipelines & Neural Network Projects

## Course Overview

This notebook covers essential MLOps practices for managing deep learning training workflows at scale. We'll explore:

1. **Deep Learning Training Pipelines** - Managing complex workflows with configuration management
2. **Compute Resource Management** - Cloud GPU optimization and cost control
3. **Experiment Tracking & Reproducibility** - MLflow, TensorBoard, and logging
4. **Neural Network Project Development** - End-to-end implementation with production practices

**Important:** This notebook uses **real datasets** from `torchvision.datasets`:
- **CIFAR-10** (Module 1) - 32×32 color images, 10 classes
- **Fashion MNIST** (Module 3) - 28×28 grayscale images, 10 classes

Both are automatically downloaded on first run.

---

# PART 1: DEEP LEARNING TRAINING PIPELINES

## Module 1.1: Managing DL Training Workflows

### What Makes a Good Training Pipeline?

A production-grade deep learning training pipeline has these characteristics:

1. **Reproducibility**: Same code + same data = same results (controlled randomness)
2. **Configurability**: Change hyperparameters without touching code
3. **Monitoring**: Real-time visibility into training progress and metrics
4. **Error Handling**: Graceful failure modes and recovery mechanisms
5. **Logging**: Complete audit trail of experiments
6. **Scalability**: Can handle different data sizes and compute resources

### Why This Matters for MLOps

In production, you'll often need to:
- Run the same model with different hyperparameters across multiple GPUs
- Track which configuration produced which results
- Reproduce results weeks or months later
- Scale training from your laptop to cloud GPUs
- Debug failing experiments without re-running for hours

A well-structured training pipeline handles all these challenges automatically.

## Module 1.2: Configuration Management with YAML and Hydra

### The Problem: Hardcoded Hyperparameters

**Bad approach (hardcoded):**
```python
learning_rate = 0.001
batch_size = 32
epochs = 100
hidden_units = 128
dropout_rate = 0.5
```

Problems:
- Can't change hyperparameters without editing code
- Easy to forget which values you used
- Hard to run experiments with different configs
- No version control for parameter changes

### The Solution: YAML Configuration Files

**Good approach (YAML config):**
```yaml
# config.yaml
model:
  hidden_units: 128
  dropout_rate: 0.5
  learning_rate: 0.001

training:
  batch_size: 32
  epochs: 100
  early_stopping_patience: 10

data:
  train_split: 0.8
  validation_split: 0.1
```

Benefits:
- Configs are version-controlled separately from code
- Easy to run multiple experiments with different configs
- Human-readable and maintainable
- Can override via command line: `python train.py learning_rate=0.01`

### Introduction to Hydra

Hydra is a framework by Facebook Research for managing configurations in Python. It handles:
- Loading YAML configs
- Command-line overrides
- Config composition and defaults
- Automatic output directory creation
- Parameter type validation

### Hands-On: Create a Configurable Training Script with Real CIFAR-10 Data

Let's build a complete example using YAML config + Hydra + MLflow logging with **real CIFAR-10 dataset**.

**Step 1: Install required packages**

In [ ]:
# Install required packages
!pip install hydra-core mlflow tensorboard torch torchvision scikit-learn pyyaml optuna -q

print("✓ All packages installed")

**Step 2: Import libraries and set up directories**

In [ ]:
import os
import yaml
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from pathlib import Path
from typing import Dict, Any
from datetime import datetime
import mlflow
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

**Step 3: Create configuration YAML files**

In [ ]:
# Create config directory
config_dir = "./configs"
os.makedirs(config_dir, exist_ok=True)

# Create base configuration for CIFAR-10
base_config = """
# Base configuration for CNN training on CIFAR-10

model:
  name: "simple_cnn"
  num_classes: 10
  input_channels: 3
  conv_filters: [32, 64, 128]
  kernel_sizes: [3, 3, 3]
  dropout_rate: 0.3
  use_batch_norm: true

training:
  batch_size: 128
  num_epochs: 30
  learning_rate: 0.001
  optimizer: "adam"
  weight_decay: 0.0001
  
  # Early stopping
  early_stopping_enabled: true
  early_stopping_patience: 5
  early_stopping_min_delta: 0.001

data:
  dataset: "cifar10"
  train_split: 0.8
  validation_split: 0.2
  num_workers: 4
  normalize: true

experiment:
  experiment_name: "cifar10_baseline"
  seed: 42
  device: "cuda"
  mixed_precision: false
  checkpoint_dir: "./checkpoints"
  log_interval: 50
"""

# Write base config
with open(f"{config_dir}/config.yaml", "w") as f:
    f.write(base_config.strip())

print("✓ Base configuration created at ./configs/config.yaml")
print("\nConfiguration content:")
print(base_config.strip())

**Step 4: Load real CIFAR-10 dataset**

We'll download CIFAR-10 directly from torchvision. On first run, this downloads ~170MB of data:

In [ ]:
# Create preprocessing pipeline
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],  # CIFAR-10 statistics
        std=[0.2470, 0.2435, 0.2616]
    )
])

print("Loading CIFAR-10 training dataset...")
cifar10_train = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

print("Loading CIFAR-10 test dataset...")
cifar10_test = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print(f"\n✓ CIFAR-10 datasets loaded successfully!")
print(f"  Training samples: {len(cifar10_train)}")
print(f"  Test samples: {len(cifar10_test)}")
print(f"  Image shape: {cifar10_train[0][0].shape}")
print(f"  Classes: {cifar10_train.classes}")

**Step 5: Visualize some CIFAR-10 samples**

In [ ]:
# Visualize samples from CIFAR-10
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()

# Denormalization function
def denormalize(img):
    img = img.clone()
    img[0] = img[0] * 0.2470 + 0.4914
    img[1] = img[1] * 0.2435 + 0.4822
    img[2] = img[2] * 0.2616 + 0.4465
    return torch.clamp(img, 0, 1)

for i in range(10):
    image, label = cifar10_train[i]
    image_vis = denormalize(image)
    axes[i].imshow(image_vis.permute(1, 2, 0))
    axes[i].set_title(cifar10_train.classes[label])
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('cifar10_samples.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ CIFAR-10 sample visualization saved as 'cifar10_samples.png'")

**Step 6: Create configurable CNN model**

In [ ]:
class ConfigurableCNN(nn.Module):
    """Configurable CNN built from configuration dictionary"""
    
    def __init__(self, config: Dict[str, Any]):
        super().__init__()
        model_cfg = config['model']
        
        # Build convolutional layers based on config
        layers = []
        in_channels = model_cfg['input_channels']
        
        for out_channels, kernel_size in zip(
            model_cfg['conv_filters'],
            model_cfg['kernel_sizes']
        ):
            layers.append(
                nn.Conv2d(
                    in_channels, 
                    out_channels, 
                    kernel_size=kernel_size,
                    padding=1
                )
            )
            
            if model_cfg.get('use_batch_norm', False):
                layers.append(nn.BatchNorm2d(out_channels))
            
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.MaxPool2d(2, 2))
            layers.append(nn.Dropout(model_cfg['dropout_rate']))
            
            in_channels = out_channels
        
        self.features = nn.Sequential(*layers)
        
        # Calculate size after convolutions (CIFAR-10 is 32x32)
        # After each max pool, size is divided by 2
        # 32 -> 16 -> 8 -> 4 for 3 conv layers
        final_spatial_size = 32 // (2 ** len(model_cfg['conv_filters']))
        final_channels = model_cfg['conv_filters'][-1]
        fc_input_size = final_channels * final_spatial_size * final_spatial_size
        
        # Fully connected layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(fc_input_size, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(model_cfg['dropout_rate']),
            nn.Linear(256, model_cfg['num_classes'])
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

print("✓ ConfigurableCNN class defined")

**Step 7: Create logging and training utilities**

In [ ]:
class TrainingLogger:
    """Handles logging to MLflow"""
    
    def __init__(self, config: Dict[str, Any], experiment_name: str):
        self.config = config
        mlflow.set_experiment(experiment_name)
        mlflow.start_run()
        mlflow.log_params(self._flatten_dict(config))
    
    @staticmethod
    def _flatten_dict(d, parent_key='', sep='/'):
        """Flatten nested dictionary for MLflow"""
        items = []
        for k, v in d.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if isinstance(v, dict):
                items.extend(
                    TrainingLogger._flatten_dict(v, new_key, sep).items()
                )
            else:
                items.append((new_key, str(v)))
        return dict(items)
    
    def log_metrics(self, epoch: int, metrics: Dict[str, float]):
        """Log metrics for an epoch"""
        for key, value in metrics.items():
            mlflow.log_metric(key, value, step=epoch)
    
    def end_run(self, final_metrics: Dict[str, float] = None):
        """End MLflow run"""
        if final_metrics:
            mlflow.log_metrics(final_metrics)
        mlflow.end_run()


class EarlyStopping:
    """Early stopping to prevent overfitting"""
    
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
    
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"Early stopping triggered after {self.counter} epochs without improvement")


def train_epoch(model, train_loader, criterion, optimizer, device, config):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = output.max(1)
        correct += predicted.eq(target).sum().item()
        total += target.size(0)
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy


def validate(model, val_loader, criterion, device):
    """Validate model"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            _, predicted = output.max(1)
            correct += predicted.eq(target).sum().item()
            total += target.size(0)
    
    avg_loss = total_loss / len(val_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

print("✓ Training utilities defined")

**Step 8: Load configuration and prepare for training**

In [ ]:
# Load configuration from YAML
with open('./configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Set random seeds for reproducibility
np.random.seed(config['experiment']['seed'])
torch.manual_seed(config['experiment']['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed(config['experiment']['seed'])

print("Configuration loaded:")
print(yaml.dump(config, default_flow_style=False))

device = torch.device(config['experiment']['device'] if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

**Step 9: Create data splits and loaders from real CIFAR-10**

In [ ]:
# Create data splits from training dataset
train_size = int(0.8 * len(cifar10_train))
val_size = len(cifar10_train) - train_size

train_data, val_data = torch.utils.data.random_split(
    cifar10_train,
    [train_size, val_size]
)

# Create data loaders
train_loader = DataLoader(
    train_data,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=0  # Set to 0 for Jupyter compatibility
)

val_loader = DataLoader(
    val_data,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    cifar10_test,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=0
)

print(f"✓ Data loaders created from real CIFAR-10:")
print(f"  Train: {len(train_data)} samples ({len(train_loader)} batches)")
print(f"  Val: {len(val_data)} samples ({len(val_loader)} batches)")
print(f"  Test: {len(cifar10_test)} samples ({len(test_loader)} batches)")

**Step 10: Create model, optimizer, and training setup**

In [ ]:
# Create model from configuration
model = ConfigurableCNN(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model created from configuration:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Set up optimization
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay']
)

# Setup early stopping
early_stopper = EarlyStopping(
    patience=config['training']['early_stopping_patience'],
    min_delta=config['training']['early_stopping_min_delta']
) if config['training']['early_stopping_enabled'] else None

print(f"\n✓ Optimizer configured: {config['training']['optimizer']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Weight decay: {config['training']['weight_decay']}")

**Step 11: Execute training loop with MLflow logging**

In [ ]:
# Initialize MLflow logger
logger = TrainingLogger(config, config['experiment']['experiment_name'])

print(f"\n✓ MLflow experiment started: {config['experiment']['experiment_name']}")
print(f"\nStarting training on CIFAR-10...\n")

best_val_accuracy = 0
training_history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(config['training']['num_epochs']):
    # Train
    train_loss, train_accuracy = train_epoch(
        model, train_loader, criterion, optimizer, device, config
    )
    
    # Validate
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)
    
    # Log metrics
    metrics = {
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'val_loss': val_loss,
        'val_accuracy': val_accuracy
    }
    logger.log_metrics(epoch, metrics)
    
    # Track history
    training_history['epoch'].append(epoch)
    training_history['train_loss'].append(train_loss)
    training_history['train_acc'].append(train_accuracy)
    training_history['val_loss'].append(val_loss)
    training_history['val_acc'].append(val_accuracy)
    
    # Print progress
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | Train: {train_accuracy:6.2f}% | Val: {val_accuracy:6.2f}% | Loss: {val_loss:.4f}")
    
    # Save best model
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        checkpoint_dir = Path(config['experiment']['checkpoint_dir'])
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_accuracy': val_accuracy,
            'config': config
        }, checkpoint_dir / 'best_model.pt')
    
    # Early stopping
    if early_stopper:
        early_stopper(val_loss)
        if early_stopper.early_stop:
            print(f"\nTraining stopped early at epoch {epoch + 1}")
            break

print(f"\nTraining completed!")
print(f"Best validation accuracy: {best_val_accuracy:.2f}%")

**Step 12: Visualize training history**

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
axes[0].plot(training_history['epoch'], training_history['train_loss'], label='Train Loss')
axes[0].plot(training_history['epoch'], training_history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy plot
axes[1].plot(training_history['epoch'], training_history['train_acc'], label='Train Accuracy')
axes[1].plot(training_history['epoch'], training_history['val_acc'], label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Training history saved")

**Step 13: End MLflow logging**

In [ ]:
# End MLflow run
logger.end_run({
    'best_val_accuracy': best_val_accuracy,
    'final_epoch': epoch + 1
})

print("✓ MLflow run completed")
print(f"\nView results with: mlflow ui")

---

# PART 2: COMPUTE RESOURCE MANAGEMENT

## Module 2.1: Cloud GPU Options & Cost Analysis

### Cloud GPU Providers

**Google Colab Pro**
- Cost: $10/month for unlimited access
- GPU: NVIDIA Tesla T4, P100, V100 (random)
- Best for: Prototyping, learning, experiments <2 hours

**AWS EC2**
- Cost: $0.35/hr (T4) to $12.48/hr (A100)
- GPU: T4, V100, A100 (flexible choice)
- Best for: Production training, flexible infrastructure

**Google Cloud (GCP)**
- Cost: $0.29/hr (T4) to $4.76/hr (A100)
- GPU: T4, V100, TPU options
- Best for: TensorFlow workloads, integrations

### Cost Estimation

In [ ]:
class CostEstimator:
    """
    Estimates training costs for different cloud GPU scenarios.
    """
    
    # Real pricing (approximate, as of 2024)
    GPU_PRICES = {
        'aws_t4': 0.35,               # $/hour
        'aws_v100': 3.06,             # $/hour
        'aws_a100': 12.48,            # $/hour
        'gcp_t4': 0.29,               # $/hour
        'gcp_v100': 2.92,             # $/hour
        'colab_pro': 10 / (24 * 30),  # $10/month
    }
    
    @staticmethod
    def estimate_training_cost(
        gpu_type: str,
        training_hours: float,
        num_experiments: int = 1,
        parallel_runs: int = 1
    ) -> Dict[str, float]:
        """
        Estimate total training cost.
        """
        
        if gpu_type not in CostEstimator.GPU_PRICES:
            raise ValueError(f"Unknown GPU type: {gpu_type}")
        
        hourly_rate = CostEstimator.GPU_PRICES[gpu_type]
        total_experiments_sequential = num_experiments / parallel_runs
        cost_per_experiment = training_hours * hourly_rate
        total_cost = cost_per_experiment * total_experiments_sequential
        
        return {
            'gpu_type': gpu_type,
            'hourly_rate': f"${hourly_rate:.2f}",
            'cost_per_experiment': f"${cost_per_experiment:.2f}",
            'total_cost': f"${total_cost:.2f}",
            'num_experiments': num_experiments,
            'parallel_runs': parallel_runs
        }
    
    @staticmethod
    def compare_gpu_options(training_hours: float, num_experiments: int = 1) -> None:
        """
        Compare costs across GPU options.
        """
        print(f"\n{'='*80}")
        print(f"Cost Comparison: {training_hours}h training × {num_experiments} experiments")
        print(f"{'='*80}\n")
        
        results = []
        for gpu_type in CostEstimator.GPU_PRICES.keys():
            result = CostEstimator.estimate_training_cost(
                gpu_type, training_hours, num_experiments
            )
            results.append(result)
        
        # Sort by cost
        results.sort(
            key=lambda x: float(x['total_cost'].strip('$'))
        )
        
        for i, result in enumerate(results, 1):
            print(f"{i}. {result['gpu_type'].upper():15s} | ", end="")
            print(f"Rate: {result['hourly_rate']:>8s} | ", end="")
            print(f"Total: {result['total_cost']:>10s}")

print("✓ CostEstimator class defined")

### Cost Estimation Examples

In [ ]:
# Scenario 1: Prototyping
print("\nSCENARIO 1: Quick Prototyping (Small experiments)")
result = CostEstimator.estimate_training_cost(
    gpu_type='aws_t4',
    training_hours=2,
    num_experiments=5
)
for key, value in result.items():
    print(f"  {key:.<35} {value}")

print("\nSCENARIO 2: Production Training (Full dataset)")
result = CostEstimator.estimate_training_cost(
    gpu_type='aws_v100',
    training_hours=10,
    num_experiments=1
)
for key, value in result.items():
    print(f"  {key:.<35} {value}")

print("\nSCENARIO 3: Parallel Hyperparameter Search")
result = CostEstimator.estimate_training_cost(
    gpu_type='aws_t4',
    training_hours=5,
    num_experiments=20,
    parallel_runs=4  # Run 4 in parallel
)
for key, value in result.items():
    print(f"  {key:.<35} {value}")

# Compare all options
CostEstimator.compare_gpu_options(training_hours=5, num_experiments=10)

---

# PART 3: NEURAL NETWORK PROJECT DEVELOPMENT

## Module 3.1: Complete Project with Real Fashion MNIST Dataset

### Real-World Project: Fashion MNIST Classification

**Why this dataset:**
- Real benchmark (28×28 grayscale, 10 clothing classes)
- Downloaded directly from torchvision
- Perfect for demonstrating neural network advantages
- Good MLOps practicum

**Project Goals:**
1. Build reproducible training pipeline
2. Compare with traditional ML baseline
3. Optimize hyperparameters
4. Version and track models

**Step 1: Load real Fashion MNIST dataset**

In [ ]:
# Create transforms for Fashion MNIST
fmnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # FMNIST statistics
])

print("Loading Fashion MNIST dataset...")
fmnist_train = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=fmnist_transform
)

fmnist_test = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=fmnist_transform
)

class_names = {
    0: 'T-shirt/top',
    1: 'Trouser',
    2: 'Pullover',
    3: 'Dress',
    4: 'Coat',
    5: 'Sandal',
    6: 'Shirt',
    7: 'Sneaker',
    8: 'Bag',
    9: 'Ankle boot'
}

print(f"✓ Fashion MNIST loaded:")
print(f"  Training samples: {len(fmnist_train)}")
print(f"  Test samples: {len(fmnist_test)}")
print(f"  Image shape: {fmnist_train[0][0].shape}")
print(f"  Classes: {list(class_names.values())}")

**Step 2: Visualize Fashion MNIST samples**

In [ ]:
# Visualize Fashion MNIST samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()

for i in range(10):
    image, label = fmnist_train[i]
    axes[i].imshow(image.squeeze(), cmap='gray')
    axes[i].set_title(class_names[label])
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('fmnist_samples.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Fashion MNIST samples saved")

**Step 3: Create Fashion MNIST configuration and model**

In [ ]:
# Create FMNIST project configuration
fmnist_config = """
# Fashion MNIST Classification Configuration

model:
  name: "fmnist_mlp"
  architecture: "mlp"
  num_classes: 10
  input_channels: 1
  input_size: 28
  hidden_units: [256, 128, 64]
  dropout_rate: 0.3
  use_batch_norm: true

training:
  batch_size: 128
  num_epochs: 50
  learning_rate: 0.001
  optimizer: "adam"
  weight_decay: 0.0001
  
  early_stopping_enabled: true
  early_stopping_patience: 10
  early_stopping_min_delta: 0.0001

data:
  dataset: "fmnist"
  train_split: 0.8
  validation_split: 0.2

experiment:
  experiment_name: "fmnist_baseline"
  seed: 42
  device: "cuda"
  checkpoint_dir: "./checkpoints/fmnist"
"""

with open("./configs/fmnist_config.yaml", "w") as f:
    f.write(fmnist_config.strip())

print("✓ Fashion MNIST configuration created")

In [ ]:
# Define Fashion MNIST classifier
class FashionMNISTClassifier(nn.Module):
    """Neural network for Fashion MNIST classification"""
    
    def __init__(self, config: Dict[str, Any]):
        super().__init__()
        model_cfg = config['model']
        
        input_size = model_cfg['input_size'] * model_cfg['input_size'] * model_cfg['input_channels']
        
        layers = [nn.Flatten()]
        prev_size = input_size
        
        for hidden_size in model_cfg['hidden_units']:
            layers.append(nn.Linear(prev_size, hidden_size))
            if model_cfg.get('use_batch_norm', False):
                layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(model_cfg['dropout_rate']))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, model_cfg['num_classes']))
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

print("✓ FashionMNISTClassifier defined")

**Step 4: Training Fashion MNIST classifier**

In [ ]:
# Load FMNIST config
with open('./configs/fmnist_config.yaml', 'r') as f:
    fmnist_cfg = yaml.safe_load(f)

# Set seeds
np.random.seed(fmnist_cfg['experiment']['seed'])
torch.manual_seed(fmnist_cfg['experiment']['seed'])

device_fmnist = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create data loaders
fmnist_train_size = int(0.8 * len(fmnist_train))
fmnist_val_size = len(fmnist_train) - fmnist_train_size

fmnist_train_data, fmnist_val_data = torch.utils.data.random_split(
    fmnist_train,
    [fmnist_train_size, fmnist_val_size]
)

fmnist_train_loader = DataLoader(
    fmnist_train_data,
    batch_size=fmnist_cfg['training']['batch_size'],
    shuffle=True,
    num_workers=0
)

fmnist_val_loader = DataLoader(
    fmnist_val_data,
    batch_size=fmnist_cfg['training']['batch_size'],
    shuffle=False,
    num_workers=0
)

fmnist_test_loader = DataLoader(
    fmnist_test,
    batch_size=fmnist_cfg['training']['batch_size'],
    shuffle=False,
    num_workers=0
)

print(f"✓ Fashion MNIST data loaders created")
print(f"  Train: {len(fmnist_train_data)}")
print(f"  Val: {len(fmnist_val_data)}")
print(f"  Test: {len(fmnist_test)}")

**Step 5: Evaluate on test set**